# AI API Toolkit 使用示例

这个notebook演示了如何使用AI API Toolkit的各种功能。

## 1. 基础设置

In [ ]:
# 导入必要的库
import asyncio
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 导入工具包
from client import AIClient
from base import Message

## 2. 基础使用

In [ ]:
# 创建客户端
client = AIClient(
    provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    enable_cache=True
)

# 创建消息
messages = [
    Message(role="system", content="You are a helpful assistant"),
    Message(role="user", content="What is Python? Answer in one sentence.")
]

# 生成响应
response = await client.complete(
    messages=messages,
    model="gpt-4o-mini",
    temperature=0.7
)

print(f"响应: {response.content}")
print(f"\n使用token: {response.usage['total_tokens']}")
print(f"延迟: {response.latency_ms:.2f}ms")
print(f"来自缓存: {response.cached}")

## 3. 测试缓存功能

In [ ]:
# 第二次请求相同的内容（应该从缓存获取）
response2 = await client.complete(
    messages=messages,
    model="gpt-4o-mini",
    temperature=0.7
)

print(f"响应: {response2.content}")
print(f"来自缓存: {response2.cached}")
print(f"延迟: {response2.latency_ms:.2f}ms")

# 查看统计信息
stats = client.get_stats()
print(f"\n统计信息:")
print(f"  总请求数: {stats['total_requests']}")
print(f"  缓存命中: {stats['cache_hits']}")
print(f"  缓存未命中: {stats['cache_misses']}")
print(f"  缓存命中率: {stats['cache_hit_rate']:.2%}")

## 4. 流式响应

In [ ]:
messages = [
    Message(role="system", content="You are a creative writer"),
    Message(role="user", content="Write a short poem about artificial intelligence.")
]

print("生成中...\n")
async for chunk in client.stream_complete(
    messages=messages,
    model="gpt-4o-mini"
):
    print(chunk, end="", flush=True)

## 5. 预取功能

In [ ]:
# 准备多个请求
messages_list = [
    [Message(role="user", content="What is AI? Answer in one sentence.")],
    [Message(role="user", content="What is Machine Learning? Answer in one sentence.")],
    [Message(role="user", content="What is Deep Learning? Answer in one sentence.")]
]

# 预取
print("预取中...")
await client.prefetch(
    messages_list=messages_list,
    model="gpt-4o-mini"
)
print("预取完成！\n")

# 从缓存获取
for i, messages in enumerate(messages_list, 1):
    response = await client.complete(
        messages=messages,
        model="gpt-4o-mini"
    )
    print(f"问题 {i}: {messages[0].content}")
    print(f"答案: {response.content}")
    print(f"来自缓存: {response.cached}")
    print()

## 6. 使用辅助方法

In [ ]:
# 使用create_messages批量创建消息
messages = client.create_messages(
    ("system", "You are a helpful coding assistant"),
    ("user", "How do I sort a list in Python?")
)

response = await client.complete(
    messages=messages,
    model="gpt-4o-mini"
)

print(response.content)

## 7. 性能对比：有缓存 vs 无缓存

In [ ]:
import time

# 清空缓存
client.clear_cache()

messages = [
    Message(role="user", content="What is the capital of France?")
]

# 第一次请求（无缓存）
start = time.time()
response1 = await client.complete(messages=messages, model="gpt-4o-mini")
time1 = (time.time() - start) * 1000

# 第二次请求（有缓存）
start = time.time()
response2 = await client.complete(messages=messages, model="gpt-4o-mini")
time2 = (time.time() - start) * 1000

print(f"第一次请求（API调用）: {time1:.2f}ms")
print(f"第二次请求（从缓存）: {time2:.2f}ms")
print(f"速度提升: {time1/time2:.1f}x")

## 8. 多轮对话

In [ ]:
# 初始化对话历史
conversation = [
    Message(role="system", content="You are a helpful assistant")
]

# 轮次1
conversation.append(Message(role="user", content="My name is Alice"))
response = await client.complete(conversation, model="gpt-4o-mini")
print(f"User: My name is Alice")
print(f"Assistant: {response.content}\n")
conversation.append(Message(role="assistant", content=response.content))

# 轮次2
conversation.append(Message(role="user", content="What's my name?"))
response = await client.complete(conversation, model="gpt-4o-mini")
print(f"User: What's my name?")
print(f"Assistant: {response.content}")

## 9. 不同温度参数的效果

In [ ]:
messages = [
    Message(role="user", content="Write a creative tagline for an AI company.")
]

temperatures = [0.0, 0.5, 1.0, 1.5]

for temp in temperatures:
    response = await client.complete(
        messages=messages,
        model="gpt-4o-mini",
        temperature=temp,
        use_cache=False  # 禁用缓存以获得不同结果
    )
    print(f"Temperature {temp}: {response.content}\n")

## 10. 最终统计

In [ ]:
stats = client.get_stats()

print("="*50)
print("最终统计信息")
print("="*50)
print(f"提供商: {stats['provider']}")
print(f"总请求数: {stats['total_requests']}")
print(f"缓存命中: {stats['cache_hits']}")
print(f"缓存未命中: {stats['cache_misses']}")
print(f"缓存命中率: {stats['cache_hit_rate']:.2%}")
print(f"总token使用: {stats['total_tokens']}")
print(f"\n缓存信息:")
print(f"  启用: {stats['cache_info']['enabled']}")
print(f"  当前大小: {stats['cache_info']['size']}")
print(f"  最大大小: {stats['cache_info']['max_size']}")
print(f"  TTL: {stats['cache_info']['ttl_seconds']}秒")